# P4-7 · MCP và hệ sinh thái tool

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

In [ ]:
!pip install -q "mcp<2"

## Lý thuyết

In [ ]:
from mcp.server.fastmcp import FastMCP
mcp = FastMCP("vi-du")

@mcp.tool()
def orders(q: str) -> str:
    """Orders."""

In [ ]:
from typing import Literal

@mcp.tool()
def search_orders(phone: str, status: Literal["đang giao", "đã giao", "đã hủy"] | None = None,
                  limit: int = 5) -> list[dict]:
    """Tìm đơn của một khách theo số điện thoại (10 chữ số).
    Lọc theo trạng thái nếu cần. Trả về tối đa `limit` đơn, mới nhất trước.
    Dùng khi khách không nhớ mã đơn. Nếu đã có mã đơn, dùng get_order."""

# model chỉ thấy tên, mô tả và schema tham số của tool (await chạy được trong notebook)
for t in await mcp.list_tools():
    print(t.name, "|", t.description.splitlines()[0], "|", t.inputSchema["properties"])

orders | Orders. | {'q': {'title': 'Q', 'type': 'string'}}
search_orders | Tìm đơn của một khách theo số điện thoại (10 chữ số). | {'phone': {'title': 'Phone', 'type': 'string'}, 'status': {'anyOf': [{'enum': ['đang giao', 'đã giao', 'đã hủy'], 'type': 'string'}, {'type': 'null'}], 'default': None, 'title': 'Status'}, 'limit': {'default': 5, 'title': 'Limit', 'type': 'integer'}}


In [ ]:
%%writefile server.py
from typing import Literal
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("shop-orders")
ORDERS = {"DH1024": {"status": "đang giao", "eta": "2026-11-06", "items": ["bàn trang điểm"], "phone": "0901234567"},
          "DH1025": {"status": "đã giao", "eta": "2026-11-02", "items": ["ghế"], "phone": "0901234567"}}

@mcp.tool()
def get_order(order_id: str) -> dict:
    """Tra một đơn hàng theo mã đơn (dạng DH + 4 chữ số, ví dụ DH1024).
    Trả về trạng thái, ngày giao dự kiến và sản phẩm. Dùng khi khách hỏi đơn của họ."""
    if order_id not in ORDERS:
        return {"error": f"Không có đơn {order_id}. Kiểm tra lại mã, dạng DH + 4 chữ số."}
    o = ORDERS[order_id]
    return {k: o[k] for k in ("status", "eta", "items")}        # không trả số điện thoại ra ngoài

@mcp.tool()
def search_orders(phone: str, status: Literal["đang giao", "đã giao", "đã hủy"] | None = None,
                  limit: int = 5) -> list[dict]:
    """Tìm đơn của một khách theo số điện thoại (10 chữ số). Lọc theo trạng thái nếu cần."""
    rows = [{"order_id": k, "status": v["status"]} for k, v in ORDERS.items()
            if v["phone"] == phone and (status is None or v["status"] == status)]
    return rows[:limit]

@mcp.resource("orders://summary")
def orders_summary() -> str:
    """Tổng quan số đơn theo trạng thái."""
    counts = {}
    for o in ORDERS.values():
        counts[o["status"]] = counts.get(o["status"], 0) + 1
    return "\n".join(f"{k}: {v}" for k, v in counts.items())

@mcp.prompt()
def weekly_report(week: str) -> str:
    """Mẫu prompt báo cáo đơn hàng theo tuần."""
    return f"Tóm tắt tình hình đơn hàng tuần {week}: số đơn mới, đơn giao trễ, khiếu nại."

if __name__ == "__main__":
    mcp.run()          # mặc định chạy qua stdio

Writing server.py


In [ ]:
%%writefile client.py
import asyncio, os, sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def main():
    params = StdioServerParameters(command=sys.executable, args=["server.py"], env=dict(os.environ))
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            print([t.name for t in tools.tools])                        # ['get_order', 'search_orders']
            r = await session.call_tool("get_order", {"order_id": "DH9"})
            print(r.content[0].text)    # {"error": "Không có đơn DH9. Kiểm tra lại mã, dạng DH + 4 chữ số."}
            res = await session.read_resource("orders://summary")
            print(res.contents[0].text)                                 # đang giao: 1 / đã giao: 1

asyncio.run(main())

Writing client.py


In [ ]:
!python client.py

['get_order', 'search_orders']
{
  "error": "Không có đơn DH9. Kiểm tra lại mã, dạng DH + 4 chữ số."
}
đang giao: 1
đã giao: 1


## Bài tập 5: MCP client {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
%%writefile check_server.py
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def main():
    params = StdioServerParameters(command="python", args=["server.py"])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            print([t.name for t in tools.tools])
            result = await session.call_tool("get_order", {"order_id": "DH1024"})
            print(result.content)

asyncio.run(main())

Writing check_server.py


In [ ]:
!python check_server.py

['get_order', 'search_orders']
[TextContent(type='text', text='{\n  "status": "đang giao",\n  "eta": "2026-11-06",\n  "items": [\n    "bàn trang điểm"\n  ]\n}', annotations=None, meta=None)]


## Bài tập 6: MCP server cho shop {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import asyncio, json
from mcp.server.fastmcp import FastMCP
mcp = FastMCP("shop")
DON = {"DH001": "đang giao", "DH002": "đã giao"}
# TODO: hai tool với docstring rõ ràng; async def main(): list_tools, call_tool

In [ ]:
@mcp.tool()
def tra_don(ma_don: str) -> str:
    """Tra trạng thái đơn hàng theo mã đơn, ví dụ DH001."""
    return DON.get(ma_don, f"Không có đơn {ma_don}. Mã đơn có dạng DH + 3 chữ số.")

@mcp.tool()
def phi_ship(tinh: str, can_nang_kg: float, hoa_toc: bool = False) -> int:
    """Tính phí giao hàng (đồng) theo tỉnh nhận và cân nặng. hoa_toc=True: giao trong ngày, gấp 1,5 lần."""
    phi = 30000 + (1000 if tinh == "Hà Nội" else 2500) * can_nang_kg
    return int(phi * (1.5 if hoa_toc else 1))

async def main():
    tools = await mcp.list_tools()
    for t in tools:
        print(t.name, "|", t.description, "|", json.dumps(t.inputSchema["properties"], ensure_ascii=False))
    kq = await mcp.call_tool("phi_ship", {"tinh": "Đà Nẵng", "can_nang_kg": 12, "hoa_toc": True})
    kq = kq[0] if isinstance(kq, tuple) else kq
    print("phi_ship →", kq[0].text)
    return tools, int(kq[0].text)

tools, phi = asyncio.run(main())

tra_don | Tra trạng thái đơn hàng theo mã đơn, ví dụ DH001. | {"ma_don": {"title": "Ma Don", "type": "string"}}
phi_ship | Tính phí giao hàng (đồng) theo tỉnh nhận và cân nặng. hoa_toc=True: giao trong ngày, gấp 1,5 lần. | {"tinh": {"title": "Tinh", "type": "string"}, "can_nang_kg": {"title": "Can Nang Kg", "type": "number"}, "hoa_toc": {"default": false, "title": "Hoa Toc", "type": "boolean"}}
phi_ship → 90000


## Bài tập 7: Kiểm tra đầu vào theo schema trước khi chạy tool {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
schema = {"type": "object",
          "properties": {"tinh": {"type": "string"}, "can_nang_kg": {"type": "number"}, "hoa_toc": {"type": "boolean"}},
          "required": ["tinh", "can_nang_kg"]}
loi_goi = [{"tinh": "Huế", "can_nang_kg": 3},
           {"tinh": "Huế"},
           {"tinh": "Huế", "can_nang_kg": "3kg"},
           {"tinh": "Huế", "can_nang_kg": 3, "giam_gia": 10},
           {"tinh": "Huế", "can_nang_kg": 2.5, "hoa_toc": "có"}]
KIEU = {"string": str, "number": (int, float), "boolean": bool}

def kiem_tra(schema, args):
    # TODO: thiếu tham số bắt buộc, tham số lạ, sai kiểu
    return []

In [ ]:
def kiem_tra(schema, args):
    loi, props = [], schema["properties"]
    for k in schema.get("required", []):
        if k not in args:
            loi.append(f"thiếu tham số bắt buộc '{k}'")
    for k, v in args.items():
        if k not in props:
            loi.append(f"tham số lạ '{k}', chỉ nhận: {', '.join(props)}")
            continue
        kieu = props[k]["type"]
        if not isinstance(v, KIEU[kieu]) or (kieu == "number" and isinstance(v, bool)):
            loi.append(f"'{k}' phải là {kieu}, nhận được {v!r}")
    return loi

for a in loi_goi:
    print(a, "→", kiem_tra(schema, a) or "hợp lệ")

{'tinh': 'Huế', 'can_nang_kg': 3} → hợp lệ
{'tinh': 'Huế'} → ["thiếu tham số bắt buộc 'can_nang_kg'"]
{'tinh': 'Huế', 'can_nang_kg': '3kg'} → ["'can_nang_kg' phải là number, nhận được '3kg'"]
{'tinh': 'Huế', 'can_nang_kg': 3, 'giam_gia': 10} → ["tham số lạ 'giam_gia', chỉ nhận: tinh, can_nang_kg, hoa_toc"]
{'tinh': 'Huế', 'can_nang_kg': 2.5, 'hoa_toc': 'có'} → ["'hoa_toc' phải là boolean, nhận được 'có'"]


## Bài tập 8: Chặn tool đọc file ra ngoài thư mục cho phép {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
from pathlib import Path
Path("docs/sub").mkdir(parents=True, exist_ok=True)
Path("docs/chinh_sach.md").write_text("Đổi trả trong 7 ngày.")
Path("docs/sub/faq.md").write_text("Giao hàng 2 ngày.")
Path(".env").write_text("API_KEY=sk-bi-mat")
BASE = Path("docs").resolve()
thu = ["chinh_sach.md", "sub/faq.md", "../.env", "sub/../../.env", "/etc/hostname", "sub/../chinh_sach.md"]

def doc_file_an_toan(duong_dan):
    # TODO: chuẩn hóa (resolve), kiểm tra nằm trong BASE, chỉ đọc file .md
    return ""

In [ ]:
def doc_file_an_toan(duong_dan):
    p = (BASE / duong_dan).resolve()
    if not p.is_relative_to(BASE) or p.suffix != ".md":
        return f"Từ chối: chỉ được đọc file .md trong thư mục docs/ (nhận '{duong_dan}')."
    if not p.exists():
        return f"Không có file '{duong_dan}' trong docs/."
    return p.read_text()

ket_qua = [doc_file_an_toan(d) for d in thu]
for d, r in zip(thu, ket_qua):
    print(f"{d:22s} → {r}")

chinh_sach.md          → Đổi trả trong 7 ngày.
sub/faq.md             → Giao hàng 2 ngày.
../.env                → Từ chối: chỉ được đọc file .md trong thư mục docs/ (nhận '../.env').
sub/../../.env         → Từ chối: chỉ được đọc file .md trong thư mục docs/ (nhận 'sub/../../.env').
/etc/hostname          → Từ chối: chỉ được đọc file .md trong thư mục docs/ (nhận '/etc/hostname').
sub/../chinh_sach.md   → Đổi trả trong 7 ngày.
